In [1]:
from __future__ import annotations
from typing import Any, Dict, Iterable, List, Optional, Tuple
from dataclasses import dataclass, field
import pandas as pd
import numpy as np
import xarray as xr
import intake
import requests
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import gsw
import sys
import os
import cmocean.cm as cm

@dataclass
class Ztake:
    cmip6_catalog: Any                     # intake-ESM catalog (top-level)
    constraints: Dict[str, Any]
    chunks: Dict[str, int] = field(default_factory=lambda: {"time": 12})
    prefer_members: Tuple[str, ...] = ("r1i1p1f1","r1i1p1f2","r1i1p1f3")
    prefer_grids:   Tuple[str, ...] = ("gn","gr","gr1")

    # internals
    filted_ds: Any = field(init=False, repr=False)
    _df: pd.DataFrame = field(init=False, repr=False)
    _best_df: pd.DataFrame = field(init=False, repr=False)

    def __post_init__(self):
        cons = dict(self.constraints)
        if "variable" in cons and "variable_id" not in cons:
            cons["variable_id"] = cons.pop("variable")
        self.filted_ds = self.cmip6_catalog.search(**cons)
        self._df = self.filted_ds.df.copy()
        
        # ensure a 'path' column exists
        if "path" not in self._df.columns:
            self._df["path"] = self._df.get("uri", pd.Series(np.arange(len(self._df)), index=self._df.index)).astype(str)

        # parse version and rank preferences
        v = self._df["version"].astype(str).str.lstrip("v")
        self._df["version_num"] = pd.to_numeric(v, errors="coerce").fillna(0).astype(int)
        self._df["member_rank"] = self._rank_by_pref(self._df["member_id"], self.prefer_members)
        self._df["grid_rank"]   = self._rank_by_pref(self._df["grid_label"], self.prefer_grids)

        # pick best row per (model, variable)
        self._best_df = (
            self._df.groupby(["source_id","variable_id"], group_keys=False)
                    .apply(self._pick_best_one)
                    .reset_index(drop=True)
        )
        # build model list once (from best_df)
        self._model_list = sorted(self._best_df["source_id"].unique().tolist())
        
    # ---------- helpers ----------
    @staticmethod
    def _rank_by_pref(s: pd.Series, prefs: Tuple[str, ...]) -> pd.Series:
        m = {v: i for i, v in enumerate(prefs)}
        return s.map(m).fillna(len(prefs)+100).astype(int)

    @staticmethod
    def _pick_best_one(g: pd.DataFrame) -> pd.DataFrame:
        return (g.sort_values(
                    by=["member_rank","grid_rank","version_num","path"],
                    ascending=[True, True, False, True],
                    kind="mergesort")
                .head(1))

    def _paths_for_choice(self, row: pd.Series) -> List[str]:
        """
        From the full filtered DF, collect *all files* that match the chosen
        model/variable + member/grid/version (so we get the full time series).
        """
        mask = (
            (self._df["source_id"]   == row["source_id"])  &
            (self._df["variable_id"] == row["variable_id"])&
            (self._df["member_id"]   == row["member_id"])  &
            (self._df["grid_label"]  == row["grid_label"]) &
            (self._df["version"]     == row["version"])
        )
        paths = self._df.loc[mask, "path"].astype(str).tolist()
        # stable order helps open_mfdataset find time in order
        paths.sort()
        if not paths:
            raise FileNotFoundError(f"No files for {row['source_id']} {row['variable_id']} {row['member_id']} {row['grid_label']} v{row['version']}")
        return paths

    # ---------- public API ----------
    @property
    def df(self) -> pd.DataFrame:
        return self._df.copy()

    @property
    def best_per_model_variable(self) -> pd.DataFrame:
        cols = ["source_id","variable_id","member_id","grid_label","version","version_num","path"]
        return self._best_df.loc[:, [c for c in cols if c in self._best_df.columns]].copy()

    def models(self) -> List[str]:
        return sorted(self._best_df["source_id"].unique())

    def variables_for(self, model: str) -> List[str]:
        return sorted(self._best_df.query("source_id == @model")["variable_id"].unique())

    def info(self) -> pd.DataFrame:
        df = self._best_df
        return (df.groupby("source_id", as_index=False)
                  .agg(n_vars=("variable_id","nunique"),
                       members=("member_id", lambda x: ",".join(sorted(pd.unique(x)))),
                       grids=("grid_label", lambda x: ",".join(sorted(pd.unique(x)))),
                       newest_version=("version_num","max"))
                  .sort_values("source_id")
                  .reset_index(drop=True))

    def open(
        self,
        variables: Optional[Iterable[str]] = None,
        time_range: Optional[Tuple[str, str]] = None,
        engine: Optional[str] = None,                 # e.g., "netcdf4" on Gadi
        decode_times: bool = True,
        drop_conflicts: bool = True,
    ) -> Dict[str, xr.Dataset]:
        """
        Open using file PATHS from the dataframe (no to_dataset_dict).
        - Concatenate files of each variable with open_mfdataset(by_coords)
        - Merge variables per model with xr.merge
        """
        sel = self._best_df
        if variables is not None:
            variables = tuple(variables)
            sel = sel[sel["variable_id"].isin(variables)]
            if sel.empty:
                raise ValueError(f"No rows match requested variables: {variables}")

        out: Dict[str, xr.Dataset] = {}
        for model, g in sel.groupby("source_id"):
            var_ds: List[xr.Dataset] = []

            for _, row in g.iterrows():
                paths = self._paths_for_choice(row)

                # If there are known “meta” vars that cause conflicts, you can drop them in preprocess
                def _pre(ds):
                    # keep only target variable + its coords to be safe
                    keep = [row["variable_id"]]
                    # try to keep coordinates present
                    keep += [c for c in ("time","lat","latitude","lon","longitude","x","y") if c in ds.variables]
                    return ds[sorted(set(v for v in keep if v in ds.variables))]

                ds = xr.open_mfdataset(
                    paths,
                    combine="by_coords",
                    parallel=True,
                    chunks=self.chunks,
                    engine=engine,
                    decode_times=decode_times,
                    preprocess=_pre,
                )

                if time_range is not None and "time" in ds.coords:
                    ds = ds.sel(time=slice(*time_range))

                var_ds.append(ds)

            if len(var_ds) == 1:
                ds_model = var_ds[0]
            else:
                ds_model = xr.merge(
                    var_ds,
                    compat="override" if drop_conflicts else "no_conflicts",
                    combine_attrs="drop_conflicts",
                )

            out[model] = ds_model

        return out


    def open_model(
        self,
        model: str,
        variables: list[str] | None = None,
        time_range: tuple[str, str] | None = None,
        engine: str | None = None,            # e.g., "netcdf4"
        decode_times: bool = True,
        drop_conflicts: bool = True,
    ) -> xr.Dataset:
        """
        Open data for a single model using file paths from the dataframe.
        Picks newest + preferred member/grid per (model, variable), concatenates
        files by coordinates, and merges variables.
    
        Returns
        -------
        xr.Dataset
        """
        # ensure model exists
        if model not in self._best_df["source_id"].unique():
            raise KeyError(f"Model '{model}' not found. Available: {sorted(self._best_df['source_id'].unique())}")
    
        sel = self._best_df[self._best_df["source_id"] == model]
        if variables is not None:
            missing = set(variables) - set(sel["variable_id"])
            if missing:
                have = sorted(sel["variable_id"].unique())
                raise ValueError(f"{model}: variables not available: {sorted(missing)}. Available: {have}")
            sel = sel[sel["variable_id"].isin(variables)]
    
        var_ds = []
        for _, row in sel.iterrows():
            paths = self._paths_for_choice(row)
    
            def _pre(ds):
                keep = [row["variable_id"]]
                keep += [c for c in ("time","lat","latitude","lon","longitude","x","y") if c in ds.variables]
                return ds[sorted(set(v for v in keep if v in ds.variables))]
    
            ds = xr.open_mfdataset(
                paths,
                combine="by_coords",
                parallel=True,
                chunks=self.chunks,
                engine=engine,
                decode_times=decode_times,
                preprocess=_pre,
            )
            if time_range is not None and "time" in ds.coords:
                ds = ds.sel(time=slice(*time_range))
            var_ds.append(ds)
    
        if len(var_ds) == 1:
            ds_model = var_ds[0]
        else:
            ds_model = xr.merge(
                var_ds,
                compat="override" if drop_conflicts else "no_conflicts",
                combine_attrs="drop_conflicts",
            )
    
        # add a bit of provenance
        rows = sel
        ds_model = ds_model.assign_attrs({
            "model": model,
            "selection_variables": ",".join(sorted(rows["variable_id"].unique())),
            "selection_member_ids": ",".join(sorted(rows["member_id"].unique())),
            "selection_grid_labels": ",".join(sorted(rows["grid_label"].unique())),
            "selection_newest_version": int(rows["version_num"].max()),
        })
        return ds_model

    def models_all(self) -> list[str]:
        """
        Return all unique model names (source_id) present in the
        filtered dataframe after applying constraints.
        """
        return sorted(self._df["source_id"].unique().tolist())

    def models_best(self) -> list[str]:
        """
        Return unique model names (source_id) present in the
        'best choice' dataframe (after newest/member/grid filtering).
        """
        return sorted(self._best_df["source_id"].unique().tolist())
    # ========= ESGF comparison utilities (embedded) =========
    @staticmethod
    def _first(x):
        return x[0] if isinstance(x, list) else x

    @staticmethod
    def _vernum(x):
        s = str(x).lstrip("v")
        try:
            return int(s)
        except Exception:
            return 0

    @staticmethod
    def _ensure_list(v):
        if v is None:
            return None
        return v if isinstance(v, (list, tuple, set)) else [v]

    @staticmethod
    def _local_latest_df(df: pd.DataFrame) -> pd.DataFrame:
        need = ["source_id","experiment_id","member_id","table_id","variable_id","grid_label","version"]
        missing = [c for c in need if c not in df.columns]
        if missing:
            raise KeyError(f"Local catalog missing columns: {missing}")
        d = df.copy()
        d["version_num"] = d["version"].apply(Ztake._vernum)
        group_cols = ["source_id","experiment_id","member_id","table_id","variable_id","grid_label"]
        return (d.sort_values(group_cols + ["version_num"])
                 .drop_duplicates(subset=group_cols, keep="last"))

    @staticmethod
    def _build_keys_from_df(df: pd.DataFrame, include_version: bool) -> set[str]:
        if include_version:
            return set(df.apply(lambda r: ".".join([
                r["source_id"], r["experiment_id"], r["member_id"],
                r["table_id"], r["variable_id"], r["grid_label"],
                f"v{str(r['version']).lstrip('v')}"
            ]), axis=1).tolist())
        else:
            return set(df.apply(lambda r: ".".join([
                r["source_id"], r["experiment_id"], r["member_id"],
                r["table_id"], r["variable_id"], r["grid_label"]
            ]), axis=1).tolist())

    @staticmethod
    def _base_and_versions_from_df(df: pd.DataFrame):
        out = {}
        for _, r in df.iterrows():
            base = (r["source_id"], r["experiment_id"], r["member_id"],
                    r["table_id"], r["variable_id"], r["grid_label"])
            ver  = f"v{str(r['version']).lstrip('v')}"
            out.setdefault(base, set()).add(ver)
        return out

    @staticmethod
    def _base_and_versions_from_docs(docs: list[dict]):
        out = {}
        f = Ztake._first
        for d in docs:
            src  = f(d.get("source_id"))
            exp  = f(d.get("experiment_id"))
            mem  = f(d.get("member_id"))
            tab  = f(d.get("table_id"))
            var  = f(d.get("variable_id"))
            grid = f(d.get("grid_label"))
            ver  = f(d.get("version"))
            if not all([src,exp,mem,tab,var,grid]):
                inst = f(d.get("instance_id"))
                if inst:
                    parts = inst.split(".")
                    if len(parts) >= 10:
                        src, exp, mem, tab, var, grid = parts[3], parts[4], parts[5], parts[6], parts[7], parts[8]
                        ver = parts[9]
            if not all([src,exp,mem,tab,var,grid]) or ver is None:
                continue
            ver = f"v{str(ver).lstrip('v')}"
            base = (src, exp, mem, tab, var, grid)
            out.setdefault(base, set()).add(ver)
        return out

    @staticmethod
    def _summarize_version_mismatch(local_map, online_map):
        result = {}
        fnum = Ztake._vernum
        for base in sorted(set(local_map) & set(online_map)):
            lv = local_map[base]
            ov = online_map[base]
            if lv == ov:
                continue
            lmax = max((fnum(v) for v in lv), default=0)
            omax = max((fnum(v) for v in ov), default=0)
            if lmax < omax: status = "local older"
            elif lmax > omax: status = "local newer"
            else: status = "different sets"
            result[".".join(base)] = {
                "local_versions": sorted(lv),
                "online_versions": sorted(ov),
                "local_max": f"v{lmax}" if lv else None,
                "online_max": f"v{omax}" if ov else None,
                "status": status,
            }
        return result

    @staticmethod
    def _ids_map_from_docs(docs: list[dict]):
        """
        Map comparison keys -> official ESGF IDs.
        key == 'source.exp.member.table.var.grid.vYYYYMMDD'
        """
        key_to_dataset_ids, key_to_instance_ids = {}, {}
        f = Ztake._first
        for d in docs:
            src  = f(d.get("source_id"))
            exp  = f(d.get("experiment_id"))
            mem  = f(d.get("member_id"))
            tab  = f(d.get("table_id"))
            var  = f(d.get("variable_id"))
            grid = f(d.get("grid_label"))
            ver  = f(d.get("version"))
            if not all([src,exp,mem,tab,var,grid,ver]):  # require all
                continue
            key = ".".join([src,exp,mem,tab,var,grid,f"v{str(ver).lstrip('v')}"])
            dsid = f(d.get("dataset_id"))
            inst = f(d.get("instance_id"))
            if dsid: key_to_dataset_ids.setdefault(key, set()).add(dsid)
            if inst: key_to_instance_ids.setdefault(key, set()).add(inst)
        return key_to_dataset_ids, key_to_instance_ids

    @staticmethod
    def _esgf_query(constraints: dict, latest: bool, limit: int, nodes: Optional[List[str]]):
        ESGF_NODES_DEFAULT = [
            "https://esgf-node.llnl.gov/esg-search/search/",
            "https://esgf-data.dkrz.de/esg-search/search/",
            "https://esgf-node.ipsl.upmc.fr/esg-search/search/",
            "https://esgf.nci.org.au/esg-search/search/",
            "https://esgf-node.ornl.gov/esg-search/search/",
        ]
        nodes = nodes or ESGF_NODES_DEFAULT
        params = {
            "project": constraints.get("project", "CMIP6"),
            "type": "Dataset",
            "latest": str(latest).lower(),
            "format": "application/solr+json",
            "limit": str(limit),
            "offset": "0",
        }
        for k in ["experiment_id","variable_id","member_id","table_id","source_id",
                  "grid_label","institution_id","activity_id"]:
            v = constraints.get(k)
            if v is not None:
                params[k] = Ztake._ensure_list(v)

        headers = {"Accept": "application/solr+json", "User-Agent": "requests-esgf-compare"}
        last_err = None
        for base in nodes:
            try:
                r = requests.get(base, params=params, headers=headers, timeout=30)
                if r.status_code != 200:
                    last_err = f"HTTP {r.status_code} from {base}"; continue
                try:
                    data = r.json()
                except Exception as e:
                    last_err = f"JSON parse failed from {base}: {e}"; continue
                docs = data.get("response", {}).get("docs", [])
                if docs:
                    return base, docs
            except requests.RequestException as e:
                last_err = e; continue
        raise RuntimeError(f"No ESGF node returned usable results. Last error: {last_err}")

    def compare_with_esgf(
        self,
        mode: str = "latest",
        limit: int = 10000,
        nodes: Optional[List[str]] = None,
        extra_constraints: Optional[Dict[str, Any]] = None,
        return_ids: bool = True,
        request_ids: bool = False,   # if True, auto-save only_online_instance_ids
    ) -> Dict[str, Any]:
        """
        Compare local intake (this Ztake's constraints) vs ESGF online.
        Returns dict with:
          node, mode, common, only_local, only_online, local_count, online_count,
          version_mismatch, local_models, online_models,
          only_online_dataset_ids, only_online_instance_ids  (when return_ids=True)
        """
        # merge base constraints + optional overrides
        cons = dict(self.constraints)
        if extra_constraints:
            cons.update(extra_constraints)
        if "variable" in cons and "variable_id" not in cons:
            cons["variable_id"] = cons.pop("variable")

        # --- Local rows (all)
        ds_local = self.cmip6_catalog.search(**cons)
        df_local_all = ds_local.df.copy()
        if not {"version","grid_label"}.issubset(df_local_all.columns):
            missing = {"version","grid_label"} - set(df_local_all.columns)
            raise KeyError(f"Local catalog missing columns: {missing}")

        local_models = sorted(df_local_all["source_id"].unique())

        # --- Local keys by mode
        if mode == "latest":
            df_local_latest = Ztake._local_latest_df(df_local_all)
            local_keys = Ztake._build_keys_from_df(df_local_latest, include_version=True)
        elif mode == "ignore_version":
            local_keys = Ztake._build_keys_from_df(df_local_all, include_version=False)
        elif mode == "all_versions":
            local_keys = Ztake._build_keys_from_df(df_local_all, include_version=True)
        else:
            raise ValueError("mode must be one of: 'latest', 'ignore_version', 'all_versions'.")

        # --- Online query
        node_used, docs = Ztake._esgf_query(cons, latest=(mode != "all_versions"),
                                            limit=limit, nodes=nodes)

        online_models = sorted({Ztake._first(d.get("source_id")) for d in docs if d.get("source_id")})
        online_base_versions = Ztake._base_and_versions_from_docs(docs)
        if mode == "ignore_version":
            online_keys = set(".".join(k) for k in online_base_versions.keys())
        else:
            online_keys = set(".".join((*k, v)) for k, vers in online_base_versions.items() for v in vers)

        # --- Version mismatch (independent of mode)
        local_base_versions = Ztake._base_and_versions_from_df(df_local_all)

        only_local  = sorted(local_keys - online_keys)
        only_online = sorted(online_keys - local_keys)
        common      = sorted(local_keys & online_keys)

        version_mismatch = Ztake._summarize_version_mismatch(local_base_versions, online_base_versions)

        result = {
            "node": node_used,
            "mode": mode,
            "common": common,
            "only_local": only_local,
            "only_online": only_online,
            "local_count": len(local_keys),
            "online_count": len(online_keys),
            "version_mismatch": version_mismatch,
            "local_models": local_models,
            "online_models": online_models,
        }

        if return_ids:
            key_to_dataset_ids, key_to_instance_ids = Ztake._ids_map_from_docs(docs)
            only_online_dataset_ids = sorted({dsid for k in only_online for dsid in key_to_dataset_ids.get(k, [])})
            only_online_instance_ids = sorted({iid for k in only_online for iid in key_to_instance_ids.get(k, [])})
            result.update({
                "only_online_dataset_ids": only_online_dataset_ids,
                "only_online_instance_ids": only_online_instance_ids,
            })

            if request_ids and only_online_instance_ids:
                # Save only the instance_ids to file, no header/footer
                Ztake.save_ids_to_file(
                    only_online_instance_ids,
                    "only_online_instance_ids.txt",
                    prefix="instance_id",
                    add_header=False,
                    add_request_note=False,
                )

        # --- Console summary (not in the file!)
        print("Node:", result["node"])
        print("Only online dataset:", len(result["only_online"]))
        if len(result["version_mismatch"]) > 0:
            print("Version mismatches:", len(result["version_mismatch"]))

        return result

    @staticmethod
    def save_ids_to_file(ids: List[str], filename: str,
                         prefix: str = "dataset_id",
                         add_header: bool = False,
                         add_request_note: bool = False) -> str:
        """
        Save IDs to file as '<prefix>=<id>' per line. Returns absolute path.
        Default writes *only* the IDs (no header/footer).
        """
        import os
        if prefix not in {"dataset_id", "instance_id"}:
            raise ValueError("prefix must be 'dataset_id' or 'instance_id'")
        abspath = os.path.abspath(filename)
        with open(abspath, "w") as f:
            for _id in ids:
                f.write(f"{prefix}={_id}\n")
        print(f"✅ Saved {len(ids)} IDs to {abspath}")
        print("Next: open https://help.nci.org.au/ and attach this file to your help request,")
        print("or email help@nci.org.au and attach the file.")
        return abspath


In [2]:
cmip6 = intake.open_esm_datastore("/g/data/dk92/catalog/v2/esm/cmip6-oi10/catalog.json")

In [3]:
constraints = dict(
    experiment_id="historical",
    variable_id="fgco2nat",
    member_id="r1i1p1f1",
    table_id="Omon"
)

# Build your Ztake
zt = Ztake(cmip6_catalog=cmip6, constraints= constraints,
           prefer_members=("r1i1p1f1","r1i1p1f2"),
           prefer_grids=("gn","gr","gr1")
          )

/jobfs/149109811.gadi-pbs/ipykernel_3614543/3098162906.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(self._pick_best_one)


In [ ]:
dif